# Part 2 — Egg-shape optimization over aspect ratio and beta

Scans the aspect ratio and the egg-shape parameter beta at constant total area and reports the shape with the largest effective-to-total area ratio (Figure 6d, 6e, 6f in the manuscript).

In [ ]:
import numpy as np
from numba import njit, prange
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
import multiprocessing as mp

def make_rect_edge(nx, ny):
    edges = np.zeros((ny, nx), dtype=np.bool_)
    edges[0, :] = True
    edges[-1, :] = True
    edges[:, 0] = True
    edges[:, -1] = True
    return edges

def make_ellipse_edge(nx, ny, aspect_ratio=1.0, beta=0.0):
    """
    Creates an egg-shaped (crashed ellipse) boundary.
    
    Parameters:
    - aspect_ratio: ry/rx ratio (y-radius / x-radius)
      aspect_ratio < 1.0: ellipse elongated in x-direction (horizontally wide)
      aspect_ratio = 1.0: circle
      aspect_ratio > 1.0: ellipse elongated in y-direction (vertically tall)
    
    - beta: egg shape parameter (0.0 to ~0.8)
      beta = 0.0: symmetric ellipse (minor axis at racket center O)
      beta > 0.0: minor axis shifted within the racket (egg shape)
      beta = (distance from racket center O to minor axis) / rx
      
      The racket center O is always at the middle of the racket's x-extent.
      Beta shifts only where the minor axis crosses, not the racket position.
      
      Higher beta = more "crashed" or asymmetric shape
      Recommended range: 0.0 ~ 0.5 (must be < 1.0)
    
    The egg shape is created by shifting where the minor (short) axis intersects
    the major (long) axis. This creates asymmetry within the racket boundary.
    
    Total area is kept approximately constant: rx * ry = base_radius^2
    
    Returns:
    - edges: Boolean array where True = boundary/outside
    - geometry_info: (racket_cx, cy, rx, ry, offset)
      - racket_cx: x-coordinate of the racket center (middle of x-extent)
      - offset: distance from racket center to minor axis position
    """
    edges = np.zeros((ny, nx), dtype=np.bool_)
    grid_cx, cy = (nx - 1) / 2, (ny - 1) / 2
    
    # Base radii - aspect_ratio = ry/rx
    # To maintain constant area: rx * ry = base_radius^2
    base_radius = min(grid_cx, cy)
    rx = base_radius / np.sqrt(aspect_ratio)  # X-axis radius
    ry = base_radius * np.sqrt(aspect_ratio)  # Y-axis radius
    
    # Offset of the minor axis from the racket center
    # beta is the ratio: offset / rx (relative to x-radius)
    # The minor axis is at racket_cx + offset
    # The racket extends from racket_cx - rx to racket_cx + rx
    offset = beta * rx  # offset within the racket
    
    # The racket center is at grid_cx (middle of the grid)
    # This ensures the racket is always centered in the display
    racket_cx = grid_cx
    
    # Minor axis position (where ry is measured from)
    minor_axis_x = racket_cx + offset
    
    # The egg shape: left side has width (rx + offset), right side has width (rx - offset)
    # from the minor axis position
    # But total width is still 2*rx, centered at racket_cx
    
    # Left edge of racket: racket_cx - rx
    # Right edge of racket: racket_cx + rx
    # Minor axis at: racket_cx + offset
    
    for y in range(ny):
        for x in range(nx):
            # Distance from minor axis in x direction, normalized
            if x < minor_axis_x:
                # Left part: from (racket_cx - rx) to minor_axis_x
                # This distance is: rx + offset
                left_width = rx + offset
                if left_width > 0:
                    # x relative to minor axis, normalized to [-1, 0]
                    dx = (x - minor_axis_x) / left_width
                else:
                    dx = -999  # Outside
            else:
                # Right part: from minor_axis_x to (racket_cx + rx)
                # This distance is: rx - offset
                right_width = rx - offset
                if right_width > 0:
                    # x relative to minor axis, normalized to [0, 1]
                    dx = (x - minor_axis_x) / right_width
                else:
                    dx = 999  # Outside
            
            dy = (y - cy) / ry if ry > 0 else 0
            
            if dx * dx + dy * dy >= 1.0:
                edges[y, x] = True
    
    return edges, (racket_cx, cy, rx, ry, offset)


def make_ellipse_edge_simple(nx, ny, aspect_ratio=1.0, beta=0.0):
    """Simple wrapper that only returns edges"""
    edges, _ = make_ellipse_edge(nx, ny, aspect_ratio, beta)
    return edges

def calculate_grid_size_for_constant_area(base_nx, base_ny, aspect_ratio, target_area):
    """
    Use fixed grid size to avoid discretization issues.
    The ellipse itself maintains constant area through rx * ry = constant.
    By keeping the grid fixed, we ensure fair comparison.
    """
    # Keep grid size constant - the ellipse area is maintained by the radius formula
    return base_nx, base_ny

# ------------------------------
# Relaxation Solver
# ------------------------------
@njit(cache=True, fastmath=True)
def _relax(height, edges, dot_y, dot_x, dot_height, max_iter=50000, tol=1e-8):
    ny, nx = height.shape
    for _ in range(max_iter):
        max_diff = 0.0
        for y in range(1, ny - 1):
            for x in range(1, nx - 1):
                if x == dot_x and y == dot_y:
                    height[y, x] = dot_height
                    continue
                if edges[y, x]:
                    height[y, x] = 0.0
                    continue
                new_val = 0.25 * (
                    height[y + 1, x] +
                    height[y - 1, x] +
                    height[y, x + 1] +
                    height[y, x - 1]
                )
                diff = abs(new_val - height[y, x])
                if diff > max_diff:
                    max_diff = diff
                height[y, x] = new_val
        if max_diff < tol:
            break
    return height

def surface(nx, ny, dot_pos, dot_height, edges):
    height = np.zeros((ny, nx), dtype=np.float64)
    dot_y, dot_x = dot_pos
    return _relax(height, edges, dot_y, dot_x, dot_height)

@njit(parallel=True, cache=True, fastmath=True)
def _calculate_forces_parallel(nx, ny, dot_height, k, edges):
    """Parallelized force calculation using numba."""
    forces = np.zeros((ny, nx), dtype=np.float64)
    
    # Calculate forces for ALL interior points
    for y in prange(ny):
        for x in range(nx):
            if edges[y, x]:
                continue
            
            # Calculate surface for this point
            height_map = np.zeros((ny, nx), dtype=np.float64)
            height_map = _relax(height_map, edges, y, x, dot_height, max_iter=50000, tol=1e-8)
            
            # Count valid neighbors and sum their heights
            neighbors_sum = 0.0
            neighbor_count = 0
            
            if y > 0:
                neighbors_sum += height_map[y-1, x]
                neighbor_count += 1
            if y < ny-1:
                neighbors_sum += height_map[y+1, x]
                neighbor_count += 1
            if x > 0:
                neighbors_sum += height_map[y, x-1]
                neighbor_count += 1
            if x < nx-1:
                neighbors_sum += height_map[y, x+1]
                neighbor_count += 1
            
            # Force calculation: expected height at center minus actual average
            forces[y, x] = k * (neighbor_count * dot_height - neighbors_sum)
    
    return forces

# ------------------------------
# Force Landscape
# ------------------------------
def force_landscape(nx, ny, dot_height=1.0, k=1.0, shape='rect', aspect_ratio=1.0, beta=0.0):
    geometry_info = None
    if shape == 'rect':
        edges = make_rect_edge(nx, ny)
    elif shape == 'ellipse':
        edges, geometry_info = make_ellipse_edge(nx, ny, aspect_ratio, beta)
    else:
        raise ValueError("shape must be 'rect' or 'ellipse'")
    
    # Use parallelized version
    forces = _calculate_forces_parallel(nx, ny, dot_height, k, edges)
    
    return forces, edges, geometry_info

# ------------------------------
# Find group edges
# ------------------------------
def group_edges(force_map, edges, alpha=1.2, method='relative'):
    """
    Find the effective hitting area based on force distribution.
    
    Parameters:
    - force_map: 2D array of forces
    - edges: boolean mask of boundary edges
    - alpha: threshold parameter
    - method: 'relative' (alpha * min_force) or 'percentile' (top alpha*100%)
    """
    # Get interior points (not edges)
    interior_mask = ~edges
    interior_forces = force_map[interior_mask]
    
    if len(interior_forces) == 0:
        return np.zeros_like(force_map, dtype=bool), np.zeros_like(force_map, dtype=bool)
    
    if method == 'percentile':
        # Use percentile-based approach: select points with force <= alpha-th percentile
        # alpha=1.2 means top 120% (but capped at 100%), so we use min(alpha*50, 100) percentile
        percentile_value = np.percentile(interior_forces, min(alpha * 50, 100))
        group_mask = (force_map <= percentile_value) & interior_mask
    else:
        # Original relative method
        min_force = np.min(interior_forces)
        group_mask = (force_map <= alpha * min_force) & interior_mask
    
    ny, nx = force_map.shape
    edge_mask = np.zeros_like(force_map, dtype=bool)
    
    for y in range(ny):
        for x in range(nx):
            if not group_mask[y, x]:
                continue
            neighbors = [
                group_mask[y-1, x] if y > 0 else False,
                group_mask[y+1, x] if y < ny-1 else False,
                group_mask[y, x-1] if x > 0 else False,
                group_mask[y, x+1] if x < nx-1 else False
            ]
            if not all(neighbors):
                edge_mask[y, x] = True
    return edge_mask, group_mask

# ------------------------------
# Optimize Racket Shape
# ------------------------------
def calculate_efficiency(base_nx, base_ny, aspect_ratio, beta=0.0, alpha=1.2, dot_height=1.0, k=1.0, method='relative', use_adaptive_grid=True, target_area=None):
    """
    Compute the effective-area-per-unit-area ratio for a given aspect_ratio and beta
    
    Parameters:
    - aspect_ratio: major/minor ratio
    - beta: egg-shape parameter (0.0 to 1.0)
    - use_adaptive_grid: If True, adjust grid size to maintain constant total area
    - target_area: Target interior area (number of dots)
    """
    if use_adaptive_grid and target_area is not None:
        nx, ny = calculate_grid_size_for_constant_area(base_nx, base_ny, aspect_ratio, target_area)
    else:
        nx, ny = base_nx, base_ny
    
    forces, edges, geometry_info = force_landscape(nx, ny, dot_height=dot_height, k=k, 
                                     shape='ellipse', aspect_ratio=aspect_ratio, beta=beta)
    edge_group_mask, group_mask = group_edges(forces, edges, alpha, method=method)
    
    # total interior area (excluding boundary)
    area_total = np.sum(~edges)
    
    # effective hitting area (Rule 2 group)
    area_effective = np.sum(group_mask)
    
    # efficiency: effective area / total area
    efficiency = area_effective / area_total if area_total > 0 else 0
    
    # diagnostic info
    interior_forces = forces[~edges]
    min_force = np.min(interior_forces) if len(interior_forces) > 0 else 0
    max_force = np.max(interior_forces) if len(interior_forces) > 0 else 0
    mean_force = np.mean(interior_forces) if len(interior_forces) > 0 else 0
    threshold = alpha * min_force
    
    return efficiency, area_total, area_effective, min_force, max_force, mean_force, threshold, nx, ny

def find_optimal_racket(nx, ny, alpha=1.2, aspect_ratios=None, betas=None, method='relative', use_adaptive_grid=True):
    """
    Test various aspect_ratio and beta values to find the optimal racket head shape
    
    Parameters:
    - nx, ny: base grid size
    - alpha: threshold for the effective-area criterion
    - aspect_ratios: list of major/minor ratios to test
    - betas: list of egg-shape parameters to test
    - method: 'relative' or 'percentile'
    - use_adaptive_grid: If True, adjust grid size to keep total area constant
    
    Returns:
    - results: list of result dictionaries, one per (aspect_ratio, beta)
    - optimal_params: the optimal (aspect_ratio, beta)
    """
    if aspect_ratios is None:
        aspect_ratios = np.linspace(1.0, 2.0, 11)
    if betas is None:
        betas = [0.0]  # Default: symmetric ellipse
    
    # Calculate target area from aspect_ratio=1.0 (circle)
    if use_adaptive_grid:
        _, edges_reference, _ = force_landscape(nx, ny, shape='ellipse', aspect_ratio=1.0, beta=0.0)
        target_area = np.sum(~edges_reference)
        print(f"Target area (from circle): {target_area} dots")
    else:
        target_area = None
    
    results = []
    
    mode_str = "Adaptive Grid (Constant Area)" if use_adaptive_grid else "Fixed Grid"
    print(f"\n{'='*110}")
    print(f"Badminton Racket Optimization Analysis - {mode_str}")
    print(f"Base: nx={nx}, ny={ny}, alpha={alpha}, method={method}")
    print(f"{'='*110}")
    print(f"{'Ratio':<8} {'Beta':<8} {'GridSize':<12} {'Total':<8} {'Effective':<10} {'Efficiency':<12} {'MinF':<8} {'MeanF':<8}")
    print(f"{'-'*110}")
    
    for ratio in aspect_ratios:
        for beta in betas:
            efficiency, total_area, effective_area, min_force, max_force, mean_force, threshold, actual_nx, actual_ny = calculate_efficiency(
                nx, ny, ratio, beta=beta, alpha=alpha, method=method, 
                use_adaptive_grid=use_adaptive_grid, target_area=target_area
            )
            
            result = {
                'aspect_ratio': ratio,
                'beta': beta,
                'grid_size': (actual_nx, actual_ny),
                'total_area': total_area,
                'effective_area': effective_area,
                'efficiency': efficiency,
                'min_force': min_force,
                'max_force': max_force,
                'mean_force': mean_force,
                'threshold': threshold
            }
            results.append(result)
            
            grid_str = f"{actual_nx}x{actual_ny}"
            print(f"{ratio:<8.3f} {beta:<8.3f} {grid_str:<12} {total_area:<8.0f} {effective_area:<10.0f} {efficiency:<12.4f} {min_force:<8.4f} {mean_force:<8.4f}")
    
    # find the optimal parameters
    optimal_idx = np.argmax([r['efficiency'] for r in results])
    optimal_ratio = results[optimal_idx]['aspect_ratio']
    optimal_beta = results[optimal_idx]['beta']
    
    print(f"{'-'*110}")
    print(f"Optimal Aspect Ratio: {optimal_ratio:.3f}, Optimal Beta: {optimal_beta:.3f}")
    print(f"Maximum Efficiency: {results[optimal_idx]['efficiency']:.4f}")
    print(f"{'='*110}\n")
    
    return results, (optimal_ratio, optimal_beta)

def visualize_optimization_results(results, param='aspect_ratio'):
    """
    Visualize the optimization results as plots
    param: 'aspect_ratio' or 'beta' - which parameter to plot on x-axis
    """
    if param == 'aspect_ratio':
        x_values = [r['aspect_ratio'] for r in results]
        x_label = 'Aspect Ratio (ry/rx)'
    else:
        x_values = [r['beta'] for r in results]
        x_label = 'Beta (Egg Shape Parameter)'
    
    efficiencies = [r['efficiency'] for r in results]
    total_areas = [r['total_area'] for r in results]
    effective_areas = [r['effective_area'] for r in results]
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # 1. Efficiency vs Parameter
    axes[0, 0].plot(x_values, efficiencies, 'b-o', linewidth=2, markersize=5)
    optimal_idx = np.argmax(efficiencies)
    axes[0, 0].plot(x_values[optimal_idx], efficiencies[optimal_idx], 
                     'r*', markersize=15, label='Optimal')
    axes[0, 0].set_xlabel(x_label)
    axes[0, 0].set_ylabel('Efficiency (Effective/Total)')
    axes[0, 0].set_title('Efficiency Ratio per Unit Area')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend()
    
    # 2. Total Area vs Parameter
    axes[0, 1].plot(x_values, total_areas, 'g-o', linewidth=2, markersize=5)
    axes[0, 1].set_xlabel(x_label)
    axes[0, 1].set_ylabel('Total Area')
    axes[0, 1].set_title('Total Area Change')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Effective Area vs Parameter
    axes[1, 0].plot(x_values, effective_areas, 'm-o', linewidth=2, markersize=5)
    axes[1, 0].set_xlabel(x_label)
    axes[1, 0].set_ylabel('Effective Area')
    axes[1, 0].set_title('Effective Area Change')
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Both areas comparison
    axes[1, 1].plot(x_values, total_areas, 'g-o', linewidth=2, 
                    markersize=5, label='Total Area')
    axes[1, 1].plot(x_values, effective_areas, 'm-o', linewidth=2, 
                    markersize=5, label='Effective Area')
    axes[1, 1].set_xlabel(x_label)
    axes[1, 1].set_ylabel('Area')
    axes[1, 1].set_title('Area Comparison')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].legend()
    
    plt.tight_layout()
    plt.show()

def visualize_beta_heatmap(results, aspect_ratios, betas):
    """
    2D heatmap showing efficiency as function of aspect_ratio and beta
    """
    # Reshape results into 2D grid
    efficiency_grid = np.zeros((len(betas), len(aspect_ratios)))
    
    for r in results:
        i = list(betas).index(r['beta']) if r['beta'] in list(betas) else -1
        j = list(aspect_ratios).index(r['aspect_ratio']) if r['aspect_ratio'] in list(aspect_ratios) else -1
        if i >= 0 and j >= 0:
            efficiency_grid[i, j] = r['efficiency']
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    im = ax.imshow(efficiency_grid, aspect='auto', origin='lower',
                   extent=[aspect_ratios[0], aspect_ratios[-1], betas[0], betas[-1]],
                   cmap='viridis')
    
    ax.set_xlabel('Aspect Ratio (ry/rx)')
    ax.set_ylabel('Beta (Egg Shape Parameter)')
    ax.set_title('Efficiency Heatmap (ry/rx: <1=horizontal, =1=circle, >1=vertical)')
    
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Efficiency')
    
    # Mark optimal point
    optimal_idx = np.argmax([r['efficiency'] for r in results])
    optimal_ratio = results[optimal_idx]['aspect_ratio']
    optimal_beta = results[optimal_idx]['beta']
    ax.plot(optimal_ratio, optimal_beta, 'r*', markersize=20, label=f'Optimal ({optimal_ratio:.2f}, {optimal_beta:.2f})')
    ax.legend()
    
    plt.tight_layout()
    plt.show()

def main():
    """
    Main entry point
    
    Beta parameter explanation:
    - Beta = 0: Perfect symmetric ellipse (minor axis at center)
    - Beta > 0: Egg-shaped (minor axis offset from center)
    - Beta = offset / rx, where offset is the distance from grid center to minor axis
    - Maximum meaningful beta is around 0.5-0.8 (beyond that the shape becomes too distorted)
    """
    # parameter setup
    nx, ny = 40, 40  # grid size
    alpha = 1.2
    method = 'relative'  # 'relative' or 'percentile'
    use_adaptive_grid = True  # keep the area constant
    
    # parameters to test
    # aspect_ratio = ry/rx, range from 1/a to a (e.g., 0.5 to 2.0)
    # <1: horizontal ellipse, =1: circle, >1: vertical ellipse
    a_max = 2.0
    aspect_ratios = np.linspace(1/a_max, a_max, 15)  # 0.5 ~ 2.0 (ry/rx)
    # Beta: 0 (symmetric ellipse) to 0.5 (max egg-shape)
    # Beta = distance_from_center_to_minor_axis / max(rx, ry)
    betas = np.linspace(0.0, 0.5, 6)  # 0.0 ~ 0.5
    
    # 1. run the optimization
    results, optimal_params = find_optimal_racket(nx, ny, alpha=alpha, 
                                                  aspect_ratios=aspect_ratios,
                                                  betas=betas,
                                                  method=method,
                                                  use_adaptive_grid=use_adaptive_grid)
    
    optimal_ratio, optimal_beta = optimal_params
    
    # 2. visualize the heatmap
    visualize_beta_heatmap(results, aspect_ratios, betas)
    
    # 3. 3D visualization for optimal shape
    print(f"Visualizing 3D force distribution of optimal racket shape (ratio={optimal_ratio:.3f}, beta={optimal_beta:.3f})...")
    forces, edges, geometry_info = force_landscape(nx, ny, dot_height=1.0, k=1.0, 
                                     shape='ellipse', aspect_ratio=optimal_ratio, beta=optimal_beta)
    forces_plot = np.ma.array(forces, mask=edges)
    edge_group_mask, group_mask = group_edges(forces, edges, alpha)
    
    cx, cy, rx, ry, offset = geometry_info
    
    X, Y = np.meshgrid(np.arange(nx), np.arange(ny))
    z_min = np.min(forces_plot) - 0.1
    
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    ax.plot_surface(X, Y, forces_plot, cmap='inferno', alpha=0.8)
    ax.scatter(X[edge_group_mask], Y[edge_group_mask], forces[edge_group_mask],
               color='cyan', s=50, label='Group Edge')
    ax.scatter(X[edges], Y[edges], z_min, color='black', s=30, label='Boundary Edge (bottom)')
    ax.scatter(X[edge_group_mask], Y[edge_group_mask], z_min, color='cyan', s=30, 
               label='Group Edge (bottom)')
    
    # Draw axes at the bottom to show racket extent and minor axis
    # cx is racket center, racket extends from cx-rx to cx+rx
    # Minor axis is at cx + offset
    
    # X-axis extent (horizontal line through cy showing racket width)
    ax.plot([cx - rx, cx + rx], [cy, cy], [z_min, z_min], 
            'r-', linewidth=3, label=f'X-axis extent (rx={rx:.1f})')
    
    # Minor axis (vertical line at cx + offset)
    ax.plot([cx + offset, cx + offset], [cy - ry, cy + ry], [z_min, z_min], 
            'b-', linewidth=3, label=f'Minor Axis (ry={ry:.1f})')
    
    # Mark the racket center (O) and minor axis position
    ax.scatter([cx], [cy], [z_min], color='green', s=100, marker='o', label='Racket Center (O)')
    if abs(offset) > 0.01:
        ax.scatter([cx + offset], [cy], [z_min], color='blue', s=100, marker='x', 
                   label=f'Minor Axis Position (β·rx={offset:.1f})')
    
    ax.set_title(f'Optimal Racket Shape (ry/rx={optimal_ratio:.3f}, Beta={optimal_beta:.3f}, alpha={alpha})')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Force')
    ax.legend(loc='upper left', fontsize=8)
    
    plt.show()
    
    # 4. Show the egg shape boundary with axes
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(~edges, cmap='gray', origin='lower')
    
    # Draw axes on 2D plot
    # Long axis (horizontal through cy) - racket extends from cx-rx to cx+rx
    ax.plot([cx - rx, cx + rx], [cy, cy], 'r-', linewidth=2, label=f'X-axis extent (rx={rx:.1f})')
    # Short axis (vertical through cx + offset)
    ax.axvline(x=cx + offset, color='b', linewidth=2, linestyle='--', label=f'Minor Axis (ry={ry:.1f})')
    # Mark racket center
    ax.plot(cx, cy, 'go', markersize=12, label='Racket Center (O)')
    if abs(offset) > 0.01:
        ax.plot(cx + offset, cy, 'bx', markersize=15, label=f'Minor Axis Position (β·rx={offset:.1f})')
    
    ax.set_title(f'Optimal Racket Shape (ry/rx={optimal_ratio:.3f}, Beta={optimal_beta:.3f})')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.legend()
    plt.show()

if __name__ == "__main__":
    main()
